# Demonstração de funcionamento
## Previsão do resultado de partidas do Brasileirão
**Inteligência Artificial II · AMF · 2026/02**

Execute as células em ordem. O treinamento utiliza 2020–2021, a seleção utiliza 2022 e o teste utiliza 2023. Esta é uma **avaliação retrospectiva**, com informações disponíveis antes de cada partida. O placar real é exibido somente para conferir a previsão.

In [1]:
from pathlib import Path
import sys
raiz = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(raiz))
from src.documentation_evidence import executar_evidencias
dados = executar_evidencias()

INÍCIO | Execução real do protocolo temporal
Treino: 689 | Validação: 363 | Teste: 362
Treino 2020–2021 → validação 2022 → teste 2023
Baseline (Majoritária): Macro F1 validação = 0.205
Regressão Logística: Macro F1 validação = 0.370
Random Forest: Macro F1 validação = 0.257
HistGradientBoosting: Macro F1 validação = 0.326
Seleção somente pela validação: Regressão Logística
CONCLUÍDO | Regressão Logística | acurácia teste 44.48% | Macro F1 0.360
Medianas calculadas apenas em treino + validação: True
Evidências salvas em reports/evidencias/


## 1. Resultados da execução
A comparação probabilística usa também as frequências de classes aprendidas no treinamento. Nenhum desses referenciais usa os resultados de 2023 para ajustar seus parâmetros.

In [2]:
import pandas as pd
from IPython.display import display, HTML
display(HTML("<style>.dataframe{font-size:16px!important}.dataframe th,.dataframe td{padding:10px 12px!important}</style>"))
linhas = []
for nome, r in [("Regressão logística", dados["teste"]),
                ("Sempre mandante", dados["baselines_teste"]["majoritaria"]),
                ("Frequências históricas", dados["baselines_teste"]["frequencias"])]:
    linhas.append({"Modelo":nome,"Acurácia":f'{r["acuracia"]:.2%}',
                   "Macro F1":f'{r["f1_macro"]:.3f}',"Log-Loss":f'{r["log_loss_val"]:.3f}',
                   "Brier":f'{r["brier_score"]:.3f}'})
display(pd.DataFrame(linhas).style.hide(axis="index"))
print("Execução:", dados["executado_em"])
print("Amostra de teste: 362 partidas; retreino final: 1.052 partidas.")

Modelo,Acurácia,Macro F1,Log-Loss,Brier
Regressão logística,44.48%,0.360,1.089,0.652
Sempre mandante,46.96%,0.213,19.117,1.061
Frequências históricas,46.96%,0.213,1.061,0.640


Execução: 2026-09-21T09:32:55-03:00
Amostra de teste: 362 partidas; retreino final: 1.052 partidas.


## 2. Previsões geradas pelo modelo
Oito primeiras partidas elegíveis do teste, em ordem cronológica; os exemplos não foram escolhidos pelo acerto. As probabilidades somam 100% antes do arredondamento.

In [3]:
from src.data_analysis import NOME_CURTO
amostra = []
for e in dados["exemplos"]:
    amostra.append({"Confronto":NOME_CURTO.get(e["mandante"],e["mandante"]) + " × " + NOME_CURTO.get(e["visitante"],e["visitante"]),
                    "P(casa)":f'{e["p_mandante"]:.1%}',"P(empate)":f'{e["p_empate"]:.1%}',
                    "P(fora)":f'{e["p_visitante"]:.1%}',"Previsão":e["previsto"],
                    "Real":e["real"],"Placar":e["placar"],"Acerto":"Sim" if e["acertou"] else "Não"})
display(pd.DataFrame(amostra).style.hide(axis="index"))
print("Partidas disputadas entre", dados["exemplos"][0]["data"][:10], "e", dados["exemplos"][-1]["data"][:10])

Confronto,P(casa),P(empate),P(fora),Previsão,Real,Placar,Acerto
Palmeiras × Cuiabá,54.6%,32.6%,12.8%,Mandante,Mandante,2 × 1,Sim
América-MG × Fluminense,39.7%,31.8%,28.5%,Mandante,Visitante,0 × 3,Não
Botafogo × São Paulo,13.0%,42.2%,44.8%,Visitante,Mandante,2 × 1,Não
Athletico-PR × Goiás,56.5%,23.2%,20.3%,Mandante,Mandante,2 × 0,Sim
Fortaleza × Internacional,34.4%,48.1%,17.5%,Empate,Empate,1 × 1,Sim
Flamengo × Coritiba,83.5%,6.6%,9.8%,Mandante,Mandante,3 × 0,Sim
Fluminense × Athletico-PR,66.2%,15.9%,17.9%,Mandante,Mandante,2 × 0,Sim
Cuiabá × Bragantino,38.8%,40.7%,20.5%,Empate,Empate,1 × 1,Sim


Partidas disputadas entre 2023-04-15 e 2023-04-22


## 3. Leitura crítica
O modelo melhora o Macro F1 em relação à classe majoritária, mas acerta menos partidas no total. A referência de frequências históricas tem Log-Loss e Brier menores. As evidências comprovam a execução e permitem inspecionar acertos e erros; não demonstram superioridade geral do modelo.

Arquivos gerados: `reports/evidencias/metricas_execucao.json` e `reports/evidencias/previsoes_exemplo.json`.